# BattleActs Actuarial Exam Preparation Products
### DATASCI 530 — Project 1
Maxine Hakimi-Bulson

This project collects publicly available information about six BattleActs
actuarial exam preparation products: Exams 5, 6C, 6U, 7, 8, and 9.
The goal is to construct datasets describing product prices, subscription
durations, advertised features, and public sample-resource links.

Data were collected on October 18, 2026. BattleActs granted written
permission for this one-time educational scraping project and for sharing
the code, dataset, and analysis on GitHub.

Collection was restricted to public product pages and sample-link metadata.
The script did not access accounts, paid course content, or customer data,
and did not download the linked sample files. It paused between products.

## Data Collection Process

Before coding, I manually inspected two products and organized their
information in a spreadsheet to determine the variables to collect.

The algorithm:
1. Opens the BattleActs homepage and the Exams menu.
2. Identifies the six exam-product links.
3. Uses a loop to click each product link and extract its information.
4. Collects sample-card descriptions, button labels, and destination URLs.
5. Converts the extracted records into two pandas DataFrames.

The products dataset contains one row per exam product. The samples
dataset contains one row per advertised sample card, linked to its product
through the product URL.

Feature indicators describe whether a feature was listed in the collected
description and overview sections. "Not listed" does not establish that
the feature is unavailable. Sample resource types are inferred from URL
extensions; file contents and link functionality were not verified.

In [1]:
# Import packages
import pandas as pd
import time
from selenium import webdriver
import re
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Open Chrome and navigate to BattleActs
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

starting_url = "https://battleactsmain.ca/v2/home"
driver.get(starting_url)

## Product Extraction Function

In [2]:
def extract_product(driver):
    # Return None when an optional text field is absent
    def get_text(xpath):
        elements = driver.find_elements("xpath", xpath)
        return elements[0].text.strip() if elements else None

    # Basic product information
    product_name = get_text("//h1")
    price_text = get_text('//h2[contains(., "$")]')

    price = (
        float(price_text.split()[0].replace("$", "").replace(",", ""))
        if price_text else None
    )
    currency = price_text.split()[-1] if price_text else None

    # Description and its bullet points
    description_parts = driver.find_elements(
        "xpath",
        '//h2[contains(., "Study Guide")]/parent::div/p'
        ' | //h2[contains(., "Study Guide")]/parent::div//li'
    )
    description = "\n".join(
        element.text.strip() for element in description_parts
    ) or None

    # Overview feature bullets
    feature_items = driver.find_elements(
        "xpath",
        '//h2[normalize-space()="Overview Of Our Features"]'
        '/parent::div/parent::div//ul/li'
    )
    features = "\n".join(
        item.text.strip() for item in feature_items
    ) or None

    # Structured fields derived from the collected text
    feature_text = features or ""
    duration_match = re.search(
        r"(\d+)-month'?s?\s+access", feature_text, re.I
    )
    extension_match = re.search(
        r"free\s+(\d+)-month\s+extension", feature_text, re.I
    )
    practice_match = re.search(
        r"(\d+)\s+original practice exams?", feature_text, re.I
    )

    # Comparison table: retain only feature names and BA descriptions
    study_details = []
    study_rows = driver.find_elements(
        "xpath",
        '//h2[normalize-space()="BattleActs Unique Study Features"]'
        '/following::table[1]/tbody/tr'
    )

    for row in study_rows:
        cells = row.find_elements("xpath", "./td")
        if len(cells) >= 2:
            study_details.append(
                cells[0].text.strip() + ": " + cells[1].text.strip()
            )

    # Introductory text and exam-specific FAQ
    intro_text = get_text(
        '//h2[normalize-space()="Pass your upcoming exam with confidence."]'
        '/following-sibling::p[1]'
    )
    exam_differences = get_text(
        '//p[contains(., "different than other")]'
        '/following-sibling::p[1]'
    )

    product_url = driver.current_url
    collection_date = pd.Timestamp.now(
        tz="America/New_York"
    ).isoformat(timespec="seconds")

    product = {
        "product_name": product_name,
        "price": price,
        "currency": currency,
        "description": description,
        "features": features,
        "duration_months": (
            int(duration_match.group(1)) if duration_match else None
        ),
        "product_url": product_url,
        "collection_date": collection_date,
        "study_features": "\n".join(study_details) or None,
        "exam_differences": exam_differences,
        "intro_text": intro_text,
        "practice_exam_count": (
            int(practice_match.group(1)) if practice_match else None
        ),
        "free_extension_months": (
            int(extension_match.group(1)) if extension_match else None
        )
    }

    # Sample cards within the section containing the sample heading
    sample_cards = driver.find_elements(
        "xpath",
        '//h2[normalize-space()="Sample our Study Platform"]'
        '/ancestor::div[contains(concat(" ", normalize-space(@class), " "),'
        ' " product-section ")][1]'
        '//div[contains(concat(" ", normalize-space(@class), " "),'
        ' " owl-item ")'
        ' and not(contains(concat(" ", normalize-space(@class), " "),'
        ' " cloned "))]'
    )

    samples = []

    for card in sample_cards:
        labels = card.find_elements("xpath", ".//p")
        controls = card.find_elements(
            "xpath", ".//a[@href] | .//button[@data-url]"
        )

        if not labels or not controls:
            raise ValueError(
                "Unexpected sample-card structure on " + product_url
            )

        control = controls[0]
        sample_url = (
            control.get_attribute("href")
            or control.get_attribute("data-url")
        )
        url_path = (sample_url or "").split("?")[0].split("#")[0].lower()

        if url_path.endswith(".pdf"):
            resource_type = "PDF"
        elif url_path.endswith((".xlsx", ".xls")):
            resource_type = "Excel"
        elif url_path.endswith(".mp4"):
            resource_type = "Video"
        else:
            resource_type = "Unknown"

        samples.append({
            "product_name": product_name,
            "sample_label": labels[0].get_attribute("textContent").strip(),
            "button_text": control.get_attribute("textContent").strip(),
            "sample_url": sample_url,
            "resource_type": resource_type,
            "product_url": product_url,
            "collection_date": collection_date
        })

    return product, samples

## Automated Navigation and Collection

In [3]:
wait = WebDriverWait(driver, 15)

# Open the homepage and collect the exam-link destinations
driver.get(starting_url)

menu_xpath = '//a[@id="navbarDropdown" and normalize-space()="Exams"]'
links_xpath = (
    '//a[contains(@class, "dropdown-item")'
    ' and contains(@href, "/v2/product/")]'
)

wait.until(EC.element_to_be_clickable((By.XPATH, menu_xpath))).click()

exam_links = driver.find_elements("xpath", links_xpath)

# Remove repeated URLs while preserving their order
exam_urls = list(dict.fromkeys(
    link.get_attribute("href") for link in exam_links
))

if len(exam_urls) != 6:
    raise ValueError(
        f"Expected 6 exam links, but found {len(exam_urls)}. Check the menu."
    )

products = []
all_samples = []

for exam_url in exam_urls:
    # Reopen the homepage and menu for each product
    driver.get(starting_url)
    wait.until(EC.element_to_be_clickable((By.XPATH, menu_xpath))).click()

    # Navigate by actually clicking the product link
    product_link_xpath = (
        '//a[contains(@class, "dropdown-item")'
        f' and @href="{exam_url}"]'
    )
    wait.until(
        EC.element_to_be_clickable((By.XPATH, product_link_xpath))
    ).click()

    wait.until(EC.url_to_be(exam_url))
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h1")))
    time.sleep(2)  # Allow the sample carousel to initialize

    product, samples = extract_product(driver)

    # Preserve completed rows as the loop progresses
    products.append(product)
    all_samples.extend(samples)

    print(
        f'{product["product_name"]}: '
        f'{product["price"]} {product["currency"]}; '
        f'{len(samples)} sample cards'
    )

    time.sleep(3)  # Pause between products

# Build the two datasets
products_df = pd.DataFrame(products)
samples_df = pd.DataFrame(all_samples)

display(products_df[
    ["product_name", "price", "currency", "duration_months"]
])
print("Total sample cards:", len(samples_df))

CAS Exam 5: 500.0 USD; 6 sample cards
CAS Exam 6C: 1000.0 CAD; 3 sample cards
CAS Exam 6U: 1000.0 USD; 6 sample cards
CAS Exam 7: 1000.0 USD; 4 sample cards
CAS Exam 8: 1000.0 USD; 3 sample cards
CAS Exam 9: 1000.0 USD; 3 sample cards


,product_name,price,currency,duration_months
0,CAS Exam 5,500.0,USD,7
1,CAS Exam 6C,1000.0,CAD,7
2,CAS Exam 6U,1000.0,USD,7
3,CAS Exam 7,1000.0,USD,7
4,CAS Exam 8,1000.0,USD,7
5,CAS Exam 9,1000.0,USD,7


Total sample cards: 25


## Data Preparation and Quality Checks

In [4]:
# Dataset sizes and duplicate checks
print("Products:", len(products_df))
print("Sample cards:", len(samples_df))
print("Duplicate product URLs:", products_df["product_url"].duplicated().sum())
print("Duplicate sample rows:", samples_df.duplicated().sum())

# Missing values, including empty text
display(
    products_df.replace(r"^\s*$", pd.NA, regex=True)
    .isna().sum().to_frame("Missing values")
)
display(
    samples_df.replace(r"^\s*$", pd.NA, regex=True)
    .isna().sum().to_frame("Missing values")
)

# Review the structured product fields
display(products_df[[
    "product_name", "practice_exam_count", "free_extension_months"
]])

# Inspect sample cards without a usable URL
missing_url = (
    samples_df["sample_url"].fillna("").str.strip().eq("")
)

display(samples_df.loc[
    missing_url,
    ["product_name", "sample_label", "button_text", "sample_url"]
])

Products: 6
Sample cards: 25
Duplicate product URLs: 0
Duplicate sample rows: 0


,Missing values
product_name,0
price,0
currency,0
description,0
features,0
duration_months,0
product_url,0
collection_date,0
study_features,0
exam_differences,0


,Missing values
product_name,0
sample_label,0
button_text,0
sample_url,1
resource_type,0
product_url,0
collection_date,0


,product_name,practice_exam_count,free_extension_months,video_lessons_listed,formula_sheet_listed,cheat_sheet_listed
0,CAS Exam 5,4,6,Yes,Yes,Yes
1,CAS Exam 6C,1,6,Not listed,Not listed,Not listed
2,CAS Exam 6U,2,6,Yes,Yes,Yes
3,CAS Exam 7,2,6,Not listed,Not listed,Not listed
4,CAS Exam 8,2,6,Not listed,Not listed,Not listed
5,CAS Exam 9,2,6,Not listed,Not listed,Not listed


,product_name,sample_label,button_text,sample_url
24,CAS Exam 9,Excel practice problems and 2 original practic...,Coming Soon,None


## Observation Count and Missing Values

In [10]:
completeness = pd.DataFrame({
    "Dataset": ["Products", "Sample cards"],
    "Observations": [len(products_df), len(samples_df)],
    "Missing values": [
        products_df.replace(r"^\s*$", pd.NA, regex=True).isna().sum().sum(),
        samples_df.replace(r"^\s*$", pd.NA, regex=True).isna().sum().sum()
    ]
})

display(completeness)

,Dataset,Observations,Missing values
0,Products,6,0
1,Sample cards,25,1


## Data Quality

The collection produced six product observations and 25 sample-card
observations. No product URLs or complete sample rows were duplicated.
All product fields were populated.

One sample URL was missing. Inspection showed that the associated Exam 9
card was labeled "Coming Soon"; this record was retained and classified
accordingly. Shared destination URLs were also retained because separate
sample cards can advertise different resources using the same link.

Exam 5 results were compared with the manually collected values:
USD 500, seven months of access, a conditional six-month extension,
and four original practice exams.

Prices were stored numerically, with currency in a separate column.
USD and CAD prices were not pooled when calculating price statistics.

## Sample Availability

In [5]:
# Classify whether each sample card has a destination
has_url = samples_df["sample_url"].fillna("").str.strip().ne("")
coming_soon = samples_df["button_text"].str.contains(
    "coming soon", case=False, na=False
)

samples_df["sample_status"] = "No URL listed"
samples_df.loc[has_url, "sample_status"] = "Link listed"
samples_df.loc[~has_url & coming_soon, "sample_status"] = "Coming soon"

# Summarize sample cards by product and status
sample_summary = pd.crosstab(
    samples_df["product_name"],
    samples_df["sample_status"]
)

display(sample_summary)

sample_status,Coming soon,Link listed
product_name,,
CAS Exam 5,0,6
CAS Exam 6C,0,3
CAS Exam 6U,0,6
CAS Exam 7,0,4
CAS Exam 8,0,3
CAS Exam 9,1,2


In [6]:
# Price statistics: keep currencies separate
price_summary = (
    products_df.groupby("currency")["price"]
    .agg(["count", "mean", "var"])
    .rename(columns={
        "count": "Products",
        "mean": "Mean price",
        "var": "Price variance"
    })
)

display(price_summary.round(2))

# Other numeric statistics: include all six products
numeric_summary = (
    products_df[
        ["duration_months", "practice_exam_count", "free_extension_months"]
    ]
    .agg(["count", "mean", "var"])
    .T
    .rename(index={
        "duration_months": "Standard access (months)",
        "practice_exam_count": "Original practice exams",
        "free_extension_months": "Conditional free extension (months)"
    }, columns={
        "count": "Observations",
        "mean": "Mean",
        "var": "Variance"
    })
)

display(numeric_summary.round(2))

,Products,Mean price,Price variance
currency,,,
CAD,1,1000.0,NaN
USD,5,900.0,50000.0


,Observations,Mean,Variance
Standard access (months),6.0,7.00,0.00
Original practice exams,6.0,2.17,0.97
Conditional free extension (months),6.0,6.00,0.00


## Discussion

The five USD-priced products have an average listed price of $900.
Exam 6C is priced separately at CAD 1,000. All six products advertise
seven months of standard access and a conditional six-month free extension.

The public sample section contains 25 advertised cards, of which 24 have
destination URLs. One Exam 9 card is marked "Coming Soon." Some cards
share a destination, so card counts do not represent counts of unique files.

These data describe one provider's public offerings at one point in time.
They do not measure course quality, exam outcomes, or independently
establish differences between BattleActs and its competitors.

In [7]:
# Save both datasets
products_df.to_csv("battleacts_products.csv", index=False)
samples_df.to_csv("battleacts_samples.csv", index=False)

In [8]:
driver.quit()